# Capítulo 10 — Optimización y paisajes

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## ¿Cuánto se gana usando la curvatura y no sólo la pendiente?

Convergencia de descenso por gradiente, gradiente con momento, BFGS y Newton
sobre la función de Rosenbrock.

La figura responde: ¿por qué la segunda derivada compensa su coste?

Ejecutar:  python fig_gradiente_newton.py

*(script original: `codigo/fig_gradiente_newton.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

from estilo_libro import C, save, use_style  # noqa: E402

use_style()


def rosenbrock(v):
    x, y = v
    return (1 - x) ** 2 + 100 * (y - x**2) ** 2


def grad(v):
    x, y = v
    return np.array([-2 * (1 - x) - 400 * x * (y - x**2), 200 * (y - x**2)])


def hess(v):
    x, y = v
    return np.array([[2 - 400 * (y - 3 * x**2), -400 * x], [-400 * x, 200]])


X0 = np.array([-1.2, 1.0])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

# --- Paisaje y trayectorias ---------------------------------------------
X, Y = np.meshgrid(np.linspace(-1.6, 1.6, 400), np.linspace(-0.6, 1.8, 400))
Z = (1 - X) ** 2 + 100 * (Y - X**2) ** 2
ax1.contour(X, Y, Z, levels=np.logspace(-0.5, 3.5, 22), colors=[C.grey],
            linewidths=0.5, alpha=0.7)
ax1.plot(1, 1, "*", color=C.ink, ms=14, zorder=6)
ax1.grid(False)

def newton_amortiguado(x0, n=60):
    """Newton con búsqueda de línea por retroceso: 12 líneas."""
    x = np.array(x0, dtype=float)
    camino = [x.copy()]
    for _ in range(n):
        g = grad(x)
        if np.linalg.norm(g) < 1e-12:
            break
        d = np.linalg.solve(hess(x), -g)
        if d @ g > 0:                    # dirección no descendente
            d = -g
        alfa = 1.0
        while rosenbrock(x + alfa * d) > rosenbrock(x) + 1e-4 * alfa * (g @ d):
            alfa *= 0.5
            if alfa < 1e-12:
                break
        x = x + alfa * d
        camino.append(x.copy())
    return camino


historiales = {}
for nombre, metodo, color, kw in [
        ("Gradiente (paso fijo)", None, C.red, {}),
        ("BFGS (cuasi-Newton)", "BFGS", C.blue, {"jac": grad}),
        ("Newton con línea", "propio", C.green, {})]:
    camino = [X0.copy()]
    if metodo is None:
        x = X0.copy()
        for _ in range(20_000):
            x = x - 1.5e-3 * grad(x)
            camino.append(x.copy())
    elif metodo == "propio":
        camino = newton_amortiguado(X0)
    else:
        minimize(rosenbrock, X0, method=metodo,
                 callback=lambda xk: camino.append(np.array(xk)), **kw)
    camino = np.array(camino)
    historiales[nombre] = (camino, color)
    paso = max(len(camino) // 400, 1)
    ax1.plot(camino[::paso, 0], camino[::paso, 1], "-", color=color, lw=1.4,
             label=f"{nombre} ({len(camino)-1} pasos)")

ax1.plot(*X0, "o", color=C.ink, ms=6)
ax1.set_xlabel("$x$"), ax1.set_ylabel("$y$")
ax1.set_title("Función de Rosenbrock: el valle del plátano")
ax1.legend(fontsize=7.6, loc="upper left")

# --- Convergencia --------------------------------------------------------
for nombre, (camino, color) in historiales.items():
    err = np.linalg.norm(camino - np.array([1.0, 1.0]), axis=1)
    ax2.semilogy(np.maximum(err, 1e-16), color=color, lw=1.6, label=nombre)
    print(f"{nombre:24s} pasos = {len(camino)-1:6d}   error final = {err[-1]:.2e}")
ax2.set_xlabel("iteración"), ax2.set_ylabel("distancia al mínimo")
ax2.set_title("Lineal, superlineal, cuadrática")
ax2.set_xlim(0, 120), ax2.set_ylim(1e-16, 5)
ax2.legend(fontsize=8)

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Por qué un ajuste puede converger a un número sin significado?

Paisaje de coste de dos ajustes: uno con parámetros identificables y otro donde
sólo una combinación está determinada.

La figura responde: ¿cómo se ve, en el paisaje, que un parámetro no es
identificable?

Ejecutar:  python fig_identificabilidad.py

*(script original: `codigo/fig_identificabilidad.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(77)

SIGMA = 0.05

# --- Caso 1: bien determinado -------------------------------------------
t1 = np.linspace(0, 6, 25)
y1 = 2.0 * np.exp(-t1 / 1.5) + r.normal(0, SIGMA, t1.size)

# --- Caso 2: dos exponenciales casi iguales (mal determinado) -----------
t2 = np.linspace(0, 3, 25)
y2 = 1.0 * np.exp(-t2 / 1.0) + 1.0 * np.exp(-t2 / 1.15) + r.normal(0, SIGMA, t2.size)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

# Paisaje 1: A y tau de una sola exponencial
A = np.linspace(1.2, 3.0, 220)
TAU = np.linspace(0.9, 2.4, 220)
AA, TT = np.meshgrid(A, TAU)
chi1 = np.zeros_like(AA)
for ti, yi in zip(t1, y1):
    chi1 += ((yi - AA * np.exp(-ti / TT)) / SIGMA) ** 2
chi1 -= chi1.min()

cs = ax1.contourf(AA, TT, np.log10(chi1 + 1), levels=25, cmap="Blues_r")
ax1.contour(AA, TT, chi1, levels=[2.30, 6.17, 11.8], colors=[C.red],
            linewidths=[1.8, 1.2, 0.9])
ax1.plot(2.0, 1.5, "*", color=C.ink, ms=14)
ax1.set_xlabel("amplitud $A$"), ax1.set_ylabel(r"tiempo $\tau$")
ax1.set_title("Identificable: el mínimo es un pozo")
ax1.grid(False)
ax1.text(1.3, 2.25, "contornos: 1, 2 y 3$\\sigma$", fontsize=8, color=C.red)

# Paisaje 2: los dos tau de la suma de exponenciales
TAU1 = np.linspace(0.5, 2.2, 220)
TAU2 = np.linspace(0.5, 2.2, 220)
T1, T2 = np.meshgrid(TAU1, TAU2)
chi2 = np.zeros_like(T1)
for ti, yi in zip(t2, y2):
    chi2 += ((yi - np.exp(-ti / T1) - np.exp(-ti / T2)) / SIGMA) ** 2
chi2 -= chi2.min()

ax2.contourf(T1, T2, np.log10(chi2 + 1), levels=25, cmap="Blues_r")
ax2.contour(T1, T2, chi2, levels=[2.30, 6.17, 11.8], colors=[C.red],
            linewidths=[1.8, 1.2, 0.9])
ax2.plot(1.0, 1.15, "*", color=C.ink, ms=14)
ax2.plot(1.15, 1.0, "*", color=C.ink, ms=14, alpha=0.6)
ax2.set_xlabel(r"$\tau_1$"), ax2.set_ylabel(r"$\tau_2$")
ax2.set_title("No identificable: el mínimo es un valle")
ax2.grid(False)
ax2.annotate("todo este valle ajusta\nigual de bien", xy=(1.5, 0.75),
             xytext=(0.6, 1.9), fontsize=8.6, color=C.red,
             arrowprops=dict(arrowstyle="->", color=C.red, lw=1.1))

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Por qué el descenso por gradiente funciona a veces y a veces no?

Tres paisajes: convexo, mal condicionado y rugoso. Sobre cada uno, la
trayectoria del descenso por gradiente desde varios puntos de partida.

La figura responde: ¿qué propiedad del paisaje decide si el problema es fácil?

Ejecutar:  python fig_paisajes.py

*(script original: `codigo/fig_paisajes.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()


def descenso(grad, x0, paso, n=300):
    x = np.array(x0, dtype=float)
    camino = [x.copy()]
    for _ in range(n):
        g = grad(x)
        x = x - paso * g
        camino.append(x.copy())
        if np.linalg.norm(g) < 1e-8:
            break
    return np.array(camino)


PAISAJES = [
    ("Convexo y bien condicionado\n$f=x^2+y^2$",
     lambda x, y: x**2 + y**2,
     lambda v: np.array([2 * v[0], 2 * v[1]]), 0.15, C.green),
    ("Convexo, mal condicionado\n$f=x^2+50y^2$",
     lambda x, y: x**2 + 50 * y**2,
     lambda v: np.array([2 * v[0], 100 * v[1]]), 0.018, C.ochre),
    ("Rugoso\n$f=x^2+y^2+3\\sin^2(3x)\\sin^2(3y)$",
     lambda x, y: x**2 + y**2 + 3 * np.sin(3 * x)**2 * np.sin(3 * y)**2,
     lambda v: np.array([
         2 * v[0] + 18 * np.sin(3 * v[0]) * np.cos(3 * v[0]) * np.sin(3 * v[1])**2,
         2 * v[1] + 18 * np.sin(3 * v[1]) * np.cos(3 * v[1]) * np.sin(3 * v[0])**2]),
     0.03, C.red),
]

fig, axes = plt.subplots(1, 3, figsize=(11.6, 4.0))
X, Y = np.meshgrid(np.linspace(-2.2, 2.2, 300), np.linspace(-2.2, 2.2, 300))

for ax, (titulo, f, grad, paso, color) in zip(axes, PAISAJES):
    Z = f(X, Y)
    ax.contourf(X, Y, Z, levels=30, cmap="Blues_r", alpha=0.55)
    ax.contour(X, Y, Z, levels=18, colors=[C.grey], linewidths=0.5, alpha=0.6)
    for x0 in [(-2.0, 1.8), (1.9, 1.5), (-1.6, -1.9), (2.0, -0.4)]:
        camino = descenso(grad, x0, paso)
        ax.plot(camino[:, 0], camino[:, 1], "-", color=color, lw=1.4, alpha=0.9)
        ax.plot(*camino[0], "o", color=C.ink, ms=4)
        ax.plot(*camino[-1], "*", color=color, ms=11, mec=C.ink, mew=0.6)
    ax.set_title(titulo, fontsize=9.5)
    ax.set_xlabel("$x$"), ax.set_ylabel("$y$")
    ax.set_aspect("equal")
    ax.grid(False)

axes[1].text(-2.1, -2.05, "zigzag: el gradiente\nno apunta al mínimo",
             fontsize=8, color=C.ochre)
axes[2].text(-2.1, -2.05, "cada salida acaba\nen un sitio distinto",
             fontsize=8, color=C.red)
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Recocido simulado: ¿por qué aceptar empeorar ayuda a mejorar?

Optimización de una función rugosa en 1D con tres temperaturas fijas y con un
enfriamiento programado, mostrando la mejor solución encontrada.

La figura responde: ¿qué papel juega exactamente la temperatura?

Ejecutar:  python fig_recocido.py

*(script original: `codigo/fig_recocido.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(1983)


def energia(x):
    """Pozo suave con mínimos locales profundos: hay barreras reales."""
    return 0.05 * x**2 + 3.0 * np.sin(2.0 * x)


def recocido(T_prog, n=40_000, x0=8.0, paso=0.35):
    x, E = x0, energia(x0)
    mejor_x, mejor_E = x, E
    camino = np.empty(n)
    for i in range(n):
        T = T_prog(i / n)
        y = x + r.normal(0, paso)
        Ey = energia(y)
        if Ey < E or (T > 0 and r.random() < np.exp(-(Ey - E) / T)):
            x, E = y, Ey
            if E < mejor_E:
                mejor_x, mejor_E = x, E
        camino[i] = x
    return camino, mejor_x, mejor_E


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2),
                               gridspec_kw={"width_ratios": [1, 1.2]})

xx = np.linspace(-10, 10, 2000)
ax1.plot(xx, energia(xx), color=C.ink, lw=1.6)
ax1.set_xlabel("$x$"), ax1.set_ylabel("energía $E(x)$")
ax1.set_title("Un paisaje con muchos mínimos locales")
x_opt = xx[np.argmin(energia(xx))]
ax1.plot(x_opt, energia(x_opt), "*", color=C.green, ms=15, zorder=5)
ax1.annotate("mínimo global", (x_opt, energia(x_opt)),
             textcoords="offset points", xytext=(10, -18), fontsize=8.4,
             color=C.green)

PROGRAMAS = [
    (lambda u: 1e-4, "$T$ = $10^{-4}$ (casi cero)", C.red),
    (lambda u: 6.0, "$T$ = 6 (muy caliente)", C.ochre),
    (lambda u: 6.0 * (1e-4 / 6.0) ** u, "enfriamiento exponencial", C.green),
]
for prog, nombre, color in PROGRAMAS:
    camino, mejor_x, mejor_E = recocido(prog)
    ax2.plot(camino, color=color, lw=0.4, alpha=0.75)
    ax1.plot(mejor_x, energia(mejor_x), "o", color=color, ms=7, zorder=6)
    print(f"{nombre:32s} mejor E = {mejor_E:+.4f}  en x = {mejor_x:+.3f}")
    ax2.plot([], [], color=color, lw=2, label=f"{nombre}: $E$ = {mejor_E:.3f}")

ax2.axhline(x_opt, color=C.ink, ls="--", lw=1.2)
ax2.text(500, x_opt + 0.35, "posición del mínimo global", fontsize=8,
         color=C.ink)
ax2.set_xlabel("iteración"), ax2.set_ylabel("$x$ visitada")
ax2.set_title("Demasiado frío se atasca; demasiado caliente no se posa")
ax2.legend(fontsize=8, loc="upper right")
ax2.set_ylim(-10, 10)

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
